In [24]:
import geopandas as gpd
import pandas as pd
from rasterstats import zonal_stats
from pyprojroot import here

In [25]:
RAW_DATA_DIR = here() / "data" / "raw"
INTERIM_DATA_DIR = here() / "data" / "interim"

In [26]:
mun = gpd.read_file(RAW_DATA_DIR/"geography/municipalities/00mun.shp").to_crs(epsg=4326)

In [27]:
# TIF files require a different treatment than shp or csv
stats = zonal_stats(
    mun,
    RAW_DATA_DIR/"resources/photovoltaic_output_tif/PVOUT.tif",
    stats=["mean", "max", "min", "std", "count"],
    all_touched=False,
)

In [28]:
solar = pd.DataFrame(stats)
solar.columns = ["pv_mean", "pv_max", "pv_min", "pv_std", "pv_pixel_count"]

In [29]:
result = pd.concat([
    mun[["CVEGEO", "CVE_ENT", "CVE_MUN", "NOMGEO"]].reset_index(drop=True),
    solar.reset_index(drop=True)
], axis=1)

In [30]:
# We verify that results are within reasonable bounds.
# pv_mean is the average photovoltaic output per hour.
print(f"PV mean range: {result['pv_mean'].min():.2f} – {result['pv_mean'].max():.2f}")
print(f"Sample:\n{result[['NOMGEO','pv_mean','pv_max']].head(10)}")

PV mean range: 3.01 – 5.32
Sample:
                      NOMGEO  pv_mean  pv_max
0             Aguascalientes    5.153   5.323
1         San José de Gracia    4.784   5.346
2  San Francisco de los Romo    5.216   5.270
3                   Tepezalá    5.170   5.321
4            Rincón de Romos    5.175   5.346
5        Pabellón de Arteaga    5.206   5.310
6                      Cosío    5.139   5.343
7                Jesús María    5.097   5.361
8                   El Llano    5.158   5.339
9                   Calvillo    4.869   5.375


In [31]:
# We annualize photovoltaic data.
result["pv_mean_annual_kwh"] = result["pv_mean"] * 365
result = result[['CVEGEO', 'pv_mean_annual_kwh']].copy()

In [33]:
result.to_csv(INTERIM_DATA_DIR/"pv_potential_by_municipality.csv", index=False, encoding="utf-8-sig")

print(f"Done. {len(result)} municipalities written.")
print(result.sample(10))

Done. 2478 municipalities written.
     CVEGEO  pv_mean_annual_kwh
861   16023         1759.665036
1940  26048         1800.180058
896   16106         1834.854982
156   07099         1249.760009
805   16104         1779.739974
507   13038         1646.515057
1414  20276         1439.195029
1109  20386         1647.609978
1384  20284         1764.045067
1400  20243         1831.570046
